# Set up

In [1]:
!pip install prince

In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

pd.set_option('display.max_columns', None)

import importlib
def reload_module(module):
    importlib.reload(module)

# Specify the directory containing the .py modules
module_dir = "modules"

# Add the directory to the Python path
sys.path.append(os.path.abspath(module_dir))

# CRLA module

In [2]:
# See crla.py for the raw code
from crla_v2 import CRLAv2 as CRLA
crla_analyzer = CRLA()

In [3]:
help(crla_analyzer)

Help on CRLAv2 in module crla_v2 object:

class CRLAv2(builtins.object)
 |  Classroom Reading Level Assessment (CRLA) data analysis class.
 |  
 |  This class handles the processing, analysis, and visualization of 
 |  reading assessment data for Grades 1-3, comparing beginning of school year (BoSY)
 |  and end of school year (EoSY) results to measure educational progress.
 |  
 |  Methods defined here:
 |  
 |  __init__(self)
 |      Initialize the CRLA class.
 |  
 |  analyze_reading_performance(self, processed_tables, combined_raw_data, combined_percentage_data, custom_weights=None)
 |      Comprehensive analysis of reading performance with data validation.
 |      
 |      Parameters
 |      ----------
 |      processed_tables : dict
 |          Dictionary of processed tables from calculate_raw_and_percentage_data
 |      combined_raw_data : dict
 |          Dictionary containing combined raw count data
 |      combined_percentage_data : dict
 |          Dictionary containing combi

## Load and preprocessing

In [4]:
%%time
# Load data from raw .CSV files of CRLA SY24-25 BoSY and EoSY
bosy_df, eosy_df = crla_analyzer.load_assessment_data(
    directory_path = 'data'
)

CPU times: user 464 ms, sys: 705 ms, total: 1.17 s
Wall time: 1.44 s


In [5]:
%%time
# GENERAL NOTE: tables refer to stand-alone dataframes
# Load relevant variables for generating tables
identifier_columns, grade_columns_dict = crla_analyzer.extract_column_structures(bosy_df)

CPU times: user 27.5 ms, sys: 314 μs, total: 27.8 ms
Wall time: 24.7 ms


In [6]:
%%time
# Generate table of unique schools and their identifiers
school_identifiers = crla_analyzer.create_school_identifier_table(
        identifier_columns, 
        assessment_dataframes=[bosy_df, eosy_df]
)

CPU times: user 40.2 ms, sys: 9.76 ms, total: 50 ms
Wall time: 39.9 ms


In [7]:
school_identifiers.head()

,Region,Division,District,School Name
School ID,,,,
131240,BARMM,Cotabato City,Cotabato City District I,Cotabato City Central Pilot School
131244,BARMM,Cotabato City,Cotabato City District II,Sero Central School
131248,BARMM,Cotabato City,Cotabato City District III,L.R. Sebastian ES
131252,BARMM,Cotabato City,Cotabato City District III,Notre Dame Village Central ES
209504,BARMM,Cotabato City,Cotabato City District III,Mandanas Elementary School


In [8]:
%%time
# Generate per grade level their corresponding concatenated BoSY & EoSY tables

grade_tables = crla_analyzer.create_grade_data_tables(
    [bosy_df, eosy_df],
    school_identifiers,
    grade_columns_dict
)

CPU times: user 12.1 s, sys: 1.81 s, total: 13.9 s
Wall time: 13.9 s


In [9]:
%%time

# Transform BoSY and EoSY grade level tables to long dataframes containing the count of learners
# that fall under the domains/subdomains languange and/or reading profiles. Count here is represented as raw
# count and percentage from total assessed in the domains/subdomains

processed_tables = crla_analyzer.calculate_raw_and_percentage_data(
    grade_tables, 
    groupby_column = "School ID"
)

CPU times: user 7min 4s, sys: 161 ms, total: 7min 4s
Wall time: 7min 4s


### In raw count

In [10]:
%%time
bosy_raw_df, eosy_raw_df = crla_analyzer.extract_data_by_period(
    processed_tables, 
    data_type='raw'
)

CPU times: user 8 μs, sys: 0 ns, total: 8 μs
Wall time: 10.7 μs


In [11]:
%%time
combined_raw_df = crla_analyzer.combine_grade_data(
    bosy_raw_df,
    eosy_raw_df,
    school_identifiers
)

CPU times: user 39.9 ms, sys: 10 ms, total: 49.9 ms
Wall time: 48.9 ms


In [12]:
combined_raw_df.keys()

dict_keys(['BoSY', 'EoSY'])

In [13]:
combined_raw_df['BoSY'].head()

,School Name,Region,Division,District,G1_Developing,G1_Grade Level,G1_Higher Emergent,G1_Lower Emergent,G1_Transitioning,G2_Fil_Developing,G2_Fil_Grade Level,G2_Fil_Higher Emergent,G2_Fil_Lower Emergent,G2_Fil_Transitioning,G2_MT_Developing,G2_MT_Grade Level,G2_MT_Higher Emergent,G2_MT_Lower Emergent,G2_MT_Transitioning,G3_Eng_Developing,G3_Eng_Grade Level,G3_Eng_Higher Emergent,G3_Eng_Lower Emergent,G3_Eng_Transitioning,G3_Fil_Developing,G3_Fil_Grade Level,G3_Fil_Higher Emergent,G3_Fil_Lower Emergent,G3_Fil_Transitioning,G3_MT_Developing,G3_MT_Grade Level,G3_MT_Higher Emergent,G3_MT_Lower Emergent,G3_MT_Transitioning
School ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100000,Balayong Elementary School,Region I,San Carlos City,San Carlos City District I,9.0,4.0,7.0,36.0,2.0,1.0,10.0,5.0,26.0,12.0,1.0,12.0,7.0,24.0,10.0,12.0,4.0,4.0,1.0,11.0,2.0,15.0,0.0,3.0,12.0,0.0,15.0,0.0,4.0,13.0
100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Bacarra I,5.0,0.0,0.0,1.0,2.0,1.0,0.0,0.0,1.0,4.0,1.0,0.0,0.0,1.0,4.0,0.0,3.0,0.0,0.0,9.0,0.0,3.0,0.0,0.0,9.0,0.0,5.0,0.0,1.0,6.0
100002,Bacarra CES,Region I,Ilocos Norte,Bacarra I,7.0,0.0,3.0,40.0,5.0,4.0,2.0,0.0,14.0,41.0,1.0,11.0,1.0,17.0,31.0,18.0,2.0,2.0,0.0,24.0,0.0,16.0,0.0,10.0,20.0,19.0,0.0,1.0,6.0,20.0
100003,Buyon ES,Region I,Ilocos Norte,Bacarra I,0.0,3.0,0.0,8.0,4.0,0.0,0.0,0.0,9.0,14.0,0.0,0.0,0.0,6.0,17.0,3.0,0.0,0.0,1.0,9.0,0.0,0.0,0.0,5.0,8.0,0.0,0.0,0.0,6.0,7.0
100004,Ganagan Elementary School,Region I,Ilocos Norte,Bacarra I,3.0,0.0,0.0,6.0,8.0,3.0,3.0,0.0,0.0,3.0,3.0,3.0,0.0,0.0,3.0,0.0,2.0,2.0,0.0,6.0,0.0,2.0,2.0,0.0,6.0,0.0,1.0,0.0,0.0,9.0


In [14]:
combined_raw_df['EoSY'].head()

,School Name,Region,Division,District,G1_Developing,G1_Grade Level,G1_Higher Emergent,G1_Lower Emergent,G1_Transitioning,G2_Fil_Developing,G2_Fil_Grade Level,G2_Fil_Higher Emergent,G2_Fil_Lower Emergent,G2_Fil_Transitioning,G2_MT_Developing,G2_MT_Grade Level,G2_MT_Higher Emergent,G2_MT_Lower Emergent,G2_MT_Transitioning,G3_Eng_Developing,G3_Eng_Grade Level,G3_Eng_Higher Emergent,G3_Eng_Lower Emergent,G3_Eng_Transitioning,G3_Fil_Developing,G3_Fil_Grade Level,G3_Fil_Higher Emergent,G3_Fil_Lower Emergent,G3_Fil_Transitioning,G3_MT_Developing,G3_MT_Grade Level,G3_MT_Higher Emergent,G3_MT_Lower Emergent,G3_MT_Transitioning
School ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100000,Balayong Elementary School,Region I,San Carlos City,San Carlos City District I,4.0,14.0,1.0,5.0,15.0,12.0,13.0,8.0,5.0,15.0,17.0,15.0,5.0,5.0,11.0,10.0,9.0,0.0,1.0,13.0,0.0,17.0,0.0,1.0,15.0,0.0,18.0,0.0,1.0,14.0
100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Bacarra I,1.0,6.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,2.0,0.0,4.0,0.0,0.0,2.0,0.0,12.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0
100002,Bacarra CES,Region I,Ilocos Norte,Bacarra I,8.0,23.0,2.0,5.0,17.0,10.0,24.0,0.0,5.0,21.0,10.0,24.0,0.0,5.0,21.0,1.0,29.0,1.0,1.0,14.0,1.0,29.0,1.0,1.0,14.0,1.0,29.0,1.0,1.0,14.0
100003,Buyon ES,Region I,Ilocos Norte,Bacarra I,0.0,7.0,0.0,0.0,6.0,0.0,11.0,0.0,0.0,12.0,0.0,11.0,0.0,0.0,12.0,0.0,4.0,0.0,0.0,9.0,0.0,7.0,0.0,2.0,4.0,0.0,8.0,0.0,1.0,4.0
100004,Ganagan Elementary School,Region I,Ilocos Norte,Bacarra I,1.0,1.0,0.0,0.0,15.0,0.0,6.0,0.0,0.0,3.0,0.0,6.0,0.0,0.0,3.0,2.0,5.0,0.0,0.0,3.0,2.0,4.0,0.0,0.0,4.0,2.0,5.0,0.0,0.0,3.0


### In percent values
The number of learners who took the CRLA differs substantially across regions. To enable fair comparisons of performance between regions, we converted raw scores to percentages. This standardization allows us to meaningfully compare aggregate regional performance despite the varying sample sizes.

In [15]:
%%time
bosy_percentage_df, eosy_percentage_df = crla_analyzer.extract_data_by_period(
    processed_tables, 
    data_type='percentages'
)

CPU times: user 15 μs, sys: 0 ns, total: 15 μs
Wall time: 18.8 μs


In [16]:
%%time

combined_percentage_df = crla_analyzer.combine_grade_data(
    bosy_percentage_df,
    eosy_percentage_df,
    school_identifiers
)

CPU times: user 84.8 ms, sys: 113 μs, total: 84.9 ms
Wall time: 83.7 ms


In [17]:
combined_percentage_df['BoSY']

,School Name,Region,Division,District,G1_Developing,G1_Grade Level,G1_Higher Emergent,G1_Lower Emergent,G1_Transitioning,G2_Fil_Developing,G2_Fil_Grade Level,G2_Fil_Higher Emergent,G2_Fil_Lower Emergent,G2_Fil_Transitioning,G2_MT_Developing,G2_MT_Grade Level,G2_MT_Higher Emergent,G2_MT_Lower Emergent,G2_MT_Transitioning,G3_Eng_Developing,G3_Eng_Grade Level,G3_Eng_Higher Emergent,G3_Eng_Lower Emergent,G3_Eng_Transitioning,G3_Fil_Developing,G3_Fil_Grade Level,G3_Fil_Higher Emergent,G3_Fil_Lower Emergent,G3_Fil_Transitioning,G3_MT_Developing,G3_MT_Grade Level,G3_MT_Higher Emergent,G3_MT_Lower Emergent,G3_MT_Transitioning
School ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100000,Balayong Elementary School,Region I,San Carlos City,San Carlos City District I,15.517241,6.896552,12.068966,62.068966,3.448276,1.851852,18.518519,9.259259,48.148148,22.222222,1.851852,22.222222,12.962963,44.444444,18.518519,37.500000,12.500000,12.500000,3.125000,34.375000,6.25,46.875000,0.0,9.375000,37.500000,0.000000,46.875000,0.000000,12.500000,40.625000
100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Bacarra I,62.500000,0.000000,0.000000,12.500000,25.000000,16.666667,0.000000,0.000000,16.666667,66.666667,16.666667,0.000000,0.000000,16.666667,66.666667,0.000000,25.000000,0.000000,0.000000,75.000000,0.00,25.000000,0.0,0.000000,75.000000,0.000000,41.666667,0.000000,8.333333,50.000000
100002,Bacarra CES,Region I,Ilocos Norte,Bacarra I,12.727273,0.000000,5.454545,72.727273,9.090909,6.557377,3.278689,0.000000,22.950820,67.213115,1.639344,18.032787,1.639344,27.868852,50.819672,39.130435,4.347826,4.347826,0.000000,52.173913,0.00,34.782609,0.0,21.739130,43.478261,41.304348,0.000000,2.173913,13.043478,43.478261
100003,Buyon ES,Region I,Ilocos Norte,Bacarra I,0.000000,20.000000,0.000000,53.333333,26.666667,0.000000,0.000000,0.000000,39.130435,60.869565,0.000000,0.000000,0.000000,26.086957,73.913043,23.076923,0.000000,0.000000,7.692308,69.230769,0.00,0.000000,0.0,38.461538,61.538462,0.000000,0.000000,0.000000,46.153846,53.846154
100004,Ganagan Elementary School,Region I,Ilocos Norte,Bacarra I,17.647059,0.000000,0.000000,35.294118,47.058824,33.333333,33.333333,0.000000,0.000000,33.333333,33.333333,33.333333,0.000000,0.000000,33.333333,0.000000,20.000000,20.000000,0.000000,60.000000,0.00,20.000000,20.0,0.000000,60.000000,0.000000,10.000000,0.000000,0.000000,90.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502889,Maligaya Integrated School,Region III,Science City of Muñoz,Munoz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502890,CURVA Integrated School,Region III,Science City of Muñoz,Munoz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502893,Mayuwi Integrated School,Region IV-A,Tayabas City,Tayabas West,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
combined_percentage_df['EoSY']

,School Name,Region,Division,District,G1_Developing,G1_Grade Level,G1_Higher Emergent,G1_Lower Emergent,G1_Transitioning,G2_Fil_Developing,G2_Fil_Grade Level,G2_Fil_Higher Emergent,G2_Fil_Lower Emergent,G2_Fil_Transitioning,G2_MT_Developing,G2_MT_Grade Level,G2_MT_Higher Emergent,G2_MT_Lower Emergent,G2_MT_Transitioning,G3_Eng_Developing,G3_Eng_Grade Level,G3_Eng_Higher Emergent,G3_Eng_Lower Emergent,G3_Eng_Transitioning,G3_Fil_Developing,G3_Fil_Grade Level,G3_Fil_Higher Emergent,G3_Fil_Lower Emergent,G3_Fil_Transitioning,G3_MT_Developing,G3_MT_Grade Level,G3_MT_Higher Emergent,G3_MT_Lower Emergent,G3_MT_Transitioning
School ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100000,Balayong Elementary School,Region I,San Carlos City,San Carlos City District I,10.256410,35.897436,2.564103,12.820513,38.461538,22.641509,24.528302,15.094340,9.433962,28.301887,32.075472,28.301887,9.433962,9.433962,20.754717,30.303030,27.272727,0.000000,3.030303,39.393939,0.000000,51.515152,0.000000,3.030303,45.454545,0.000000,54.545455,0.000000,3.030303,42.424242
100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Bacarra I,14.285714,85.714286,0.000000,0.000000,0.000000,0.000000,66.666667,0.000000,0.000000,33.333333,0.000000,66.666667,0.000000,0.000000,33.333333,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000
100002,Bacarra CES,Region I,Ilocos Norte,Bacarra I,14.545455,41.818182,3.636364,9.090909,30.909091,16.666667,40.000000,0.000000,8.333333,35.000000,16.666667,40.000000,0.000000,8.333333,35.000000,2.173913,63.043478,2.173913,2.173913,30.434783,2.173913,63.043478,2.173913,2.173913,30.434783,2.173913,63.043478,2.173913,2.173913,30.434783
100003,Buyon ES,Region I,Ilocos Norte,Bacarra I,0.000000,53.846154,0.000000,0.000000,46.153846,0.000000,47.826087,0.000000,0.000000,52.173913,0.000000,47.826087,0.000000,0.000000,52.173913,0.000000,30.769231,0.000000,0.000000,69.230769,0.000000,53.846154,0.000000,15.384615,30.769231,0.000000,61.538462,0.000000,7.692308,30.769231
100004,Ganagan Elementary School,Region I,Ilocos Norte,Bacarra I,5.882353,5.882353,0.000000,0.000000,88.235294,0.000000,66.666667,0.000000,0.000000,33.333333,0.000000,66.666667,0.000000,0.000000,33.333333,20.000000,50.000000,0.000000,0.000000,30.000000,20.000000,40.000000,0.000000,0.000000,40.000000,20.000000,50.000000,0.000000,0.000000,30.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502889,Maligaya Integrated School,Region III,Science City of Muñoz,Munoz,14.754098,55.737705,14.754098,4.918033,9.836066,23.595506,53.932584,7.865169,0.000000,14.606742,23.595506,53.932584,7.865169,0.000000,14.606742,1.538462,23.076923,0.000000,6.153846,69.230769,0.000000,56.923077,0.000000,12.307692,30.769231,0.000000,56.923077,0.000000,12.307692,30.769231
502890,CURVA Integrated School,Region III,Science City of Muñoz,Munoz,0.000000,47.500000,0.000000,35.000000,17.500000,11.627907,67.441860,0.000000,0.000000,20.930233,11.627907,67.441860,0.000000,0.000000,20.930233,8.000000,48.000000,4.000000,0.000000,40.000000,10.000000,54.000000,0.000000,0.000000,36.000000,10.000000,54.000000,0.000000,0.000000,36.000000
502893,Mayuwi Integrated School,Region IV-A,Tayabas City,Tayabas West,0.000000,21.052632,0.000000,21.052632,57.894737,0.000000,30.769231,0.000000,23.076923,46.153846,0.000000,30.769231,0.000000,23.076923,46.153846,20.000000,20.000000,0.000000,30.000000,30.000000,0.000000,20.000000,0.000000,40.000000,40.000000,0.000000,20.000000,0.000000,40.000000,40.000000


In [19]:
%%time
default_analysis_results = crla_analyzer.analyze_reading_performance(
        processed_tables,
        combined_raw_df,
        combined_percentage_df
)

CPU times: user 1min 1s, sys: 751 ms, total: 1min 2s
Wall time: 1min 1s


In [20]:
pca_weights = crla_analyzer.optimize_weights_from_pca()

PCA-derived weights based on valid schools:
  Developing: 5
  Transitioning: 14
  Higher Emergent: 9
  Lower Emergent: 0
  Grade Level: 100


In [21]:
pca_weights

{'Developing': 5,
 'Transitioning': 14,
 'Higher Emergent': 9,
 'Lower Emergent': 0,
 'Grade Level': 100}

In [22]:
%%time
pca_analysis_results = crla_analyzer.analyze_reading_performance(
        processed_tables,
        combined_raw_df,
        combined_percentage_df,
        pca_weights
)

Performance calculated using weights: Developing: 5, Transitioning: 14, Higher Emergent: 9, Lower Emergent: 0, Grade Level: 100
CPU times: user 1min 18s, sys: 670 ms, total: 1min 19s
Wall time: 1min 18s


In [23]:
%%time
info = crla_analyzer.get_pca_components_info()

=== PCA Component Analysis ===
Number of components: 1
Number of features: 30

Component 1 Loadings (Top 5 highest absolute value):
  G3_Eng_Grade Level: 0.2846
  G3_Fil_Grade Level: 0.2837
  G3_MT_Grade Level: 0.2792
  G2_MT_Grade Level: 0.2662
  G2_Fil_Grade Level: 0.2639

Average Component 1 Loadings by Category:
  Grade Level: 0.2684
  Lower Emergent: -0.1803
  Developing: -0.1572
  Higher Emergent: -0.1413
  Transitioning: -0.1189

Explained variance ratio: 0.2753
CPU times: user 4.71 ms, sys: 52 μs, total: 4.76 ms
Wall time: 4.4 ms


In [24]:
info

,G1_Developing,G2_Fil_Developing,G2_MT_Developing,G3_Eng_Developing,G3_Fil_Developing,G3_MT_Developing,G1_Transitioning,G2_Fil_Transitioning,G2_MT_Transitioning,G3_Eng_Transitioning,G3_Fil_Transitioning,G3_MT_Transitioning,G1_Higher Emergent,G2_Fil_Higher Emergent,G2_MT_Higher Emergent,G3_Eng_Higher Emergent,G3_Fil_Higher Emergent,G3_MT_Higher Emergent,G1_Lower Emergent,G2_Fil_Lower Emergent,G2_MT_Lower Emergent,G3_Eng_Lower Emergent,G3_Fil_Lower Emergent,G3_MT_Lower Emergent,G1_Grade Level,G2_Fil_Grade Level,G2_MT_Grade Level,G3_Eng_Grade Level,G3_Fil_Grade Level,G3_MT_Grade Level
0,-0.109445,-0.159373,-0.158754,-0.175137,-0.172287,-0.168461,-0.028852,-0.11182,-0.112986,-0.136792,-0.161864,-0.160854,-0.119215,-0.133341,-0.134093,-0.179641,-0.143054,-0.138484,-0.186373,-0.183991,-0.184755,-0.155502,-0.185145,-0.185995,0.232881,0.263876,0.266234,0.284575,0.283673,0.279172


In [25]:
validation = pca_analysis_results['validation']
valid_count = validation['valid_for_progress'].sum()
total_count = len(validation)

print("=== CRLA Data Quality Summary ===")
print(f"Total schools: {total_count}")
print(f"Schools with BoSY data only: {(validation['has_bosy_data'] & ~validation['has_eosy_data']).sum()}")
print(f"Schools with EoSY data only: {(~validation['has_bosy_data'] & validation['has_eosy_data']).sum()}")

if 'count_mismatch' in validation.columns:
    print(f"Schools with significant student count differences: {validation['count_mismatch'].sum()}")

=== CRLA Data Quality Summary ===
Total schools: 37137
Schools with BoSY data only: 0
Schools with EoSY data only: 0
Schools with significant student count differences: 853


In [26]:
results = pca_analysis_results['results']
results.head(10)

,School Name,Region,Division,District,Has BoSY Data,Has EoSY Data,Student Count Mismatch,Valid for Progress Analysis,BoSY Performance,EoSY Performance,Progress Score,BoSY Developing %,BoSY G1 Developing,BoSY G2 Fil Developing,BoSY G2 MT Developing,BoSY G3 Eng Developing,BoSY G3 Fil Developing,BoSY G3 MT Developing,BoSY Transitioning %,BoSY G1 Transitioning,BoSY G2 Fil Transitioning,BoSY G2 MT Transitioning,BoSY G3 Eng Transitioning,BoSY G3 Fil Transitioning,BoSY G3 MT Transitioning,BoSY Higher Emergent %,BoSY G1 Higher Emergent,BoSY G2 Fil Higher Emergent,BoSY G2 MT Higher Emergent,BoSY G3 Eng Higher Emergent,BoSY G3 Fil Higher Emergent,BoSY G3 MT Higher Emergent,BoSY Lower Emergent %,BoSY G1 Lower Emergent,BoSY G2 Fil Lower Emergent,BoSY G2 MT Lower Emergent,BoSY G3 Eng Lower Emergent,BoSY G3 Fil Lower Emergent,BoSY G3 MT Lower Emergent,BoSY Grade Level %,BoSY G1 Grade Level,BoSY G2 Fil Grade Level,BoSY G2 MT Grade Level,BoSY G3 Eng Grade Level,BoSY G3 Fil Grade Level,BoSY G3 MT Grade Level,BoSY Developing Weighted,BoSY Transitioning Weighted,BoSY Higher Emergent Weighted,BoSY Lower Emergent Weighted,BoSY Grade Level Weighted,EoSY Developing %,EoSY G1 Developing,EoSY G2 Fil Developing,EoSY G2 MT Developing,EoSY G3 Eng Developing,EoSY G3 Fil Developing,EoSY G3 MT Developing,EoSY Transitioning %,EoSY G1 Transitioning,EoSY G2 Fil Transitioning,EoSY G2 MT Transitioning,EoSY G3 Eng Transitioning,EoSY G3 Fil Transitioning,EoSY G3 MT Transitioning,EoSY Higher Emergent %,EoSY G1 Higher Emergent,EoSY G2 Fil Higher Emergent,EoSY G2 MT Higher Emergent,EoSY G3 Eng Higher Emergent,EoSY G3 Fil Higher Emergent,EoSY G3 MT Higher Emergent,EoSY Lower Emergent %,EoSY G1 Lower Emergent,EoSY G2 Fil Lower Emergent,EoSY G2 MT Lower Emergent,EoSY G3 Eng Lower Emergent,EoSY G3 Fil Lower Emergent,EoSY G3 MT Lower Emergent,EoSY Grade Level %,EoSY G1 Grade Level,EoSY G2 Fil Grade Level,EoSY G2 MT Grade Level,EoSY G3 Eng Grade Level,EoSY G3 Fil Grade Level,EoSY G3 MT Grade Level,EoSY Developing Weighted,EoSY Transitioning Weighted,EoSY Higher Emergent Weighted,EoSY Lower Emergent Weighted,EoSY Grade Level Weighted
131072,Gapok Central School,Region XII,Sultan Kudarat,Kulaman II,True,True,False,True,2.222307,17.096974,14.874667,11.767818,7.142857,17.647059,14.705882,6.666667,13.333333,11.111111,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,25.068472,40.476190,26.470588,32.352941,13.333333,20.000000,17.777778,63.163710,52.380952,55.882353,52.941176,80.000000,66.666667,71.111111,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.588391,0.000000,2.256162,0.0,0.000000,15.362589,7.142857,11.764706,8.823529,26.666667,26.666667,11.111111,47.335823,19.047619,64.705882,64.705882,51.111111,40.000000,44.444444,4.074074,0.000000,0.000000,0.000000,6.666667,6.666667,11.111111,19.105198,54.761905,8.823529,8.823529,4.444444,15.555556,22.222222,14.122316,19.047619,14.705882,17.647059,11.111111,11.111111,11.111111,0.768129,6.627015,0.366667,0.0,14.122316
131073,Kapatagan Elementary School,Region XII,Sultan Kudarat,Kulaman II,True,True,False,True,40.945050,32.932277,-8.012773,6.250000,37.500000,0.000000,0.000000,0.000000,0.000000,0.000000,40.239846,18.750000,47.058824,47.058824,42.857143,42.857143,42.857143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.046569,18.750000,11.764706,11.764706,0.000000,0.000000,0.000000,46.463585,25.000000,41.176471,41.176471,57.142857,57.142857,57.142857,0.312500,5.633578,0.000000,0.0,46.463585,13.286648,13.333333,11.764706,11.764706,14.285714,14.285714,14.285714,1.960784,0.000000,5.882353,5.882353,0.000000,0.000000,0.000000,13.286648,13.333333,11.764706,11.764706,14.285714,14.285714,14.285714,31.447246,46.666667,35.294118,35.294118,28.571429,14.285714,28.571429,40.018674,26.666667,35.294118,35.294118,42.857143,57.142857,42.857143,0.664332,0.274510,1.195798,0.0,40.018674
131074,Lagubang Elementary School,Region XII,Sultan Kudarat,Kulaman II,True,True,False,True,1.943468,15.327433,13.383965,7.857

In [33]:
default_results = default_analysis_results['results']
default_results.head(10)

,School Name,Region,Division,District,Has BoSY Data,Has EoSY Data,Student Count Mismatch,Valid for Progress Analysis,BoSY Performance,EoSY Performance,Progress Score,BoSY Developing %,BoSY G1 Developing,BoSY G2 Fil Developing,BoSY G2 MT Developing,BoSY G3 Eng Developing,BoSY G3 Fil Developing,BoSY G3 MT Developing,BoSY Transitioning %,BoSY G1 Transitioning,BoSY G2 Fil Transitioning,BoSY G2 MT Transitioning,BoSY G3 Eng Transitioning,BoSY G3 Fil Transitioning,BoSY G3 MT Transitioning,BoSY Higher Emergent %,BoSY G1 Higher Emergent,BoSY G2 Fil Higher Emergent,BoSY G2 MT Higher Emergent,BoSY G3 Eng Higher Emergent,BoSY G3 Fil Higher Emergent,BoSY G3 MT Higher Emergent,BoSY Lower Emergent %,BoSY G1 Lower Emergent,BoSY G2 Fil Lower Emergent,BoSY G2 MT Lower Emergent,BoSY G3 Eng Lower Emergent,BoSY G3 Fil Lower Emergent,BoSY G3 MT Lower Emergent,BoSY Grade Level %,BoSY G1 Grade Level,BoSY G2 Fil Grade Level,BoSY G2 MT Grade Level,BoSY G3 Eng Grade Level,BoSY G3 Fil Grade Level,BoSY G3 MT Grade Level,EoSY Developing %,EoSY G1 Developing,EoSY G2 Fil Developing,EoSY G2 MT Developing,EoSY G3 Eng Developing,EoSY G3 Fil Developing,EoSY G3 MT Developing,EoSY Transitioning %,EoSY G1 Transitioning,EoSY G2 Fil Transitioning,EoSY G2 MT Transitioning,EoSY G3 Eng Transitioning,EoSY G3 Fil Transitioning,EoSY G3 MT Transitioning,EoSY Higher Emergent %,EoSY G1 Higher Emergent,EoSY G2 Fil Higher Emergent,EoSY G2 MT Higher Emergent,EoSY G3 Eng Higher Emergent,EoSY G3 Fil Higher Emergent,EoSY G3 MT Higher Emergent,EoSY Lower Emergent %,EoSY G1 Lower Emergent,EoSY G2 Fil Lower Emergent,EoSY G2 MT Lower Emergent,EoSY G3 Eng Lower Emergent,EoSY G3 Fil Lower Emergent,EoSY G3 MT Lower Emergent,EoSY Grade Level %,EoSY G1 Grade Level,EoSY G2 Fil Grade Level,EoSY G2 MT Grade Level,EoSY G3 Eng Grade Level,EoSY G3 Fil Grade Level,EoSY G3 MT Grade Level
131072,Gapok Central School,Region XII,Sultan Kudarat,Kulaman II,True,True,False,True,20.0,20.0,-3.552714e-15,11.767818,7.142857,17.647059,14.705882,6.666667,13.333333,11.111111,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,25.068472,40.476190,26.470588,32.352941,13.333333,20.000000,17.777778,63.163710,52.380952,55.882353,52.941176,80.000000,66.666667,71.111111,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,15.362589,7.142857,11.764706,8.823529,26.666667,26.666667,11.111111,47.335823,19.047619,64.705882,64.705882,51.111111,40.000000,44.444444,4.074074,0.000000,0.000000,0.000000,6.666667,6.666667,11.111111,19.105198,54.761905,8.823529,8.823529,4.444444,15.555556,22.222222,14.122316,19.047619,14.705882,17.647059,11.111111,11.111111,11.111111
131073,Kapatagan Elementary School,Region XII,Sultan Kudarat,Kulaman II,True,True,False,True,20.0,20.0,3.552714e-15,6.250000,37.500000,0.000000,0.000000,0.000000,0.000000,0.000000,40.239846,18.750000,47.058824,47.058824,42.857143,42.857143,42.857143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.046569,18.750000,11.764706,11.764706,0.000000,0.000000,0.000000,46.463585,25.000000,41.176471,41.176471,57.142857,57.142857,57.142857,13.286648,13.333333,11.764706,11.764706,14.285714,14.285714,14.285714,1.960784,0.000000,5.882353,5.882353,0.000000,0.000000,0.000000,13.286648,13.333333,11.764706,11.764706,14.285714,14.285714,14.285714,31.447246,46.666667,35.294118,35.294118,28.571429,14.285714,28.571429,40.018674,26.666667,35.294118,35.294118,42.857143,57.142857,42.857143
131074,Lagubang Elementary School,Region XII,Sultan Kudarat,Kulaman II,True,True,False,True,20.0,20.0,3.552714e-15,7.857143,0.000000,3.333333,6.666667,8.571429,14.285714,14.285714,10.932118,17.021277,0.000000,0.000000,14.285714,17.142857,17.142857,6.269841,0.000000,13.333333,10.000000,14.285714,0.000000,0.000000,74.940898,82.978723,83.333333,83.333333,62.857143,68.571429,68.571429,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.197255,32.653061,38.709677,38.709677,0.000000,5.555556,5.555556,11.581694,40.816327,3.225806,3.225806,11.111111,5.555556,5.5555

In [27]:
results[results['Valid for Progress Analysis'] == False].shape

(853, 91)

In [28]:
results[results['Valid for Progress Analysis'] == False].Region.value_counts()

Region
Region V       179
Region IV-B    130
Region III      73
Region VI       65
CAR             56
Region VIII     52
Region XII      45
Region VII      42
Region I        34
Region IV-A     33
Region II       32
Region XI       31
Region IX       28
Region X        26
CARAGA          17
NCR              6
BARMM            4
Name: count, dtype: int64

In [29]:
results[results['Valid for Progress Analysis'] == True].Region.value_counts()

Region
Region VIII    3603
Region VI      3360
Region V       2990
Region III     2968
Region VII     2924
Region IV-A    2730
Region I       2380
Region II      2184
Region X       2125
Region IX      2108
Region IV-B    1784
Region XI      1718
Region XII     1703
CARAGA         1691
CAR            1476
NCR             515
BARMM            25
Name: count, dtype: int64

In [30]:
print("=== Top 5 Schools by Progress (Valid Schools Only) ===")
valid_schools = results[results['Valid for Progress Analysis']]
top_progress = valid_schools.sort_values('Progress Score', ascending=False).head(5)
for idx, row in top_progress.iterrows():
    print(f"School ID: {idx}")
    print(f"  School Name: {row['School Name']}")
    print(f"  Region: {row['Region']}")
    print(f"  Progress Score: {row['Progress Score']:.2f}")
    print(f"  BoSY to EoSY Change: {row['BoSY Performance']:.2f} → {row['EoSY Performance']:.2f}")
    print()

=== Top 5 Schools by Progress (Valid Schools Only) ===
School ID: 136960
  School Name: Mantapay Elementary School
  Region: Region XI
  Progress Score: 78.12
  BoSY to EoSY Change: 0.00 → 78.12

School ID: 100204
  School Name: Nagabungan ES
  Region: Region I
  Progress Score: 76.13
  BoSY to EoSY Change: 2.00 → 78.12

School ID: 136975
  School Name: Damayon IP School
  Region: CARAGA
  Progress Score: 75.76
  BoSY to EoSY Change: 2.36 → 78.12

School ID: 219515
  School Name: Matso-Tokiyas Elementary School (formerly Ambanglo PS)
  Region: CAR
  Progress Score: 75.52
  BoSY to EoSY Change: 2.60 → 78.12

School ID: 135874
  School Name: Nagacadan ES
  Region: CAR
  Progress Score: 74.62
  BoSY to EoSY Change: 3.51 → 78.12



In [34]:
print("=== Top 5 Schools by Progress (Valid Schools Only) - Default Weights ===")
valid_schools = default_results[default_results['Valid for Progress Analysis']]
top_progress = valid_schools.sort_values('Progress Score', ascending=False).head(5)
for idx, row in top_progress.iterrows():
    print(f"School ID: {idx}")
    print(f"  School Name: {row['School Name']}")
    print(f"  Region: {row['Region']}")
    print(f"  Progress Score: {row['Progress Score']:.2f}")
    print(f"  BoSY to EoSY Change: {row['BoSY Performance']:.2f} → {row['EoSY Performance']:.2f}")
    print()

=== Top 5 Schools by Progress (Valid Schools Only) - Default Weights ===
School ID: 115320
  School Name: Indag-an ES
  Region: Region VI
  Progress Score: 10.00
  BoSY to EoSY Change: 10.00 → 20.00

School ID: 137123
  School Name: Cambayang Elementary School
  Region: Region IV-B
  Progress Score: 6.67
  BoSY to EoSY Change: 13.33 → 20.00

School ID: 115775
  School Name: Buri PS
  Region: Region VI
  Progress Score: 6.67
  BoSY to EoSY Change: 13.33 → 20.00

School ID: 106501
  School Name: Laungcupang ES
  Region: Region III
  Progress Score: 6.67
  BoSY to EoSY Change: 13.33 → 20.00

School ID: 212047
  School Name: SIKAT PANLABUAN PS
  Region: CARAGA
  Progress Score: 6.67
  BoSY to EoSY Change: 13.33 → 20.00



In [31]:
pca_analysis_results['results'].to_csv("data/crla_progress_score.pca_derived.csv")

In [32]:
default_analysis_results['results'].to_csv("data/crla_progress_score.default.csv")

# Optimizing Reading Proficiency Weights: PCA-Based Analysis

## Introduction

Classroom Reading Level Assessment (CRLA) performance scores can be calculated through various weighting methodologies for reading proficiency categories. This analysis employs Principal Component Analysis (PCA) to derive empirically-based weights from assessment data, specifically using only schools with valid data in both assessment periods. The research compares standard unweighted approaches with the data-driven PCA methodology to determine optimal category weighting.

## Principal Component Analysis

### Mathematical Framework

Principal Component Analysis operates by identifying orthogonal directions of maximum variance in multivariate data. The technique relies on Singular Value Decomposition (SVD), a fundamental matrix factorization method that decomposes the standardized data matrix X into:

$$\text{X = UΣV}^{\text{T}}$$

Where:
- U contains left singular vectors (eigenvectors of XX^T)
- Σ contains singular values (square roots of eigenvalues)
- V contains right singular vectors (eigenvectors of X^TX)

The principal components are given by XV, representing projections of the original data onto the eigenvectors.

### From Data to Weights: The PCA Process

1. **Standardization**: The data is first standardized using Z-scores:
   $$Z = \frac{X - \mu}{\sigma}$$
   where $X$ is the original data, $\mu$ is the mean, and $\sigma$ is the standard deviation.

2. **Covariance Matrix**: A covariance matrix $C$ of the standardized data is calculated, which measures how variables change together.

3. **Eigendecomposition**: The eigenvectors and eigenvalues of this matrix are found:
   $$C \cdot v = \lambda \cdot v$$
   where $v$ is an eigenvector and $\lambda$ is the corresponding eigenvalue.

4. **Factor Loadings**: The loadings are essentially the correlation between the original variables and the principal components:
   $$L_{ij} = v_{ij} \cdot \sqrt{\lambda_j}$$
   where $L_{ij}$ is the loading of variable $i$ on component $j$.

5. **Weight Conversion**: The loadings from the first principal component (the one explaining the most variance) are taken, properly oriented, and scaled to a 0-100 range:
   $$W_i = 100 \cdot \frac{L_i - \min(L)}{\max(L) - \min(L)}$$
   where $W_i$ is the weight for category $i$.

The first principal component captures the maximum possible variance, making its loadings particularly valuable as weights. These loadings reflect the relative importance of each reading proficiency category in differentiating school performance based on empirical patterns rather than theoretical assumptions.

## Standard vs. PCA-Derived Weights

The comparison between standard unweighted metrics and the PCA-derived weights reveals striking differences:

| Reading Proficiency Level | Standard Approach | PCA-Derived Weights |
|---------------------------|-------------------|---------------------|
| Developing               | 100               | 5                   |
| Transitioning            | 100               | 14                  |
| Higher Emergent          | 100               | 9                   |
| Lower Emergent           | 100               | 0                   |
| Grade Level              | 100               | 100                 |

## Key Insights

The PCA-derived weights provide several important insights into reading assessment:

1. **Hierarchical Importance**: Not all reading proficiency categories contribute equally to performance differentiation. "Grade Level" proficiency appears to be the most discriminating factor in the dataset.

2. **Counter-Intuitive Contributions**: Despite common assumptions, the "Developing" category (typically considered high proficiency) contributes minimally to differentiation between schools.

3. **Statistical Redundancy**: The near-zero weight for "Lower Emergent" suggests this category may provide little unique information about school performance beyond what other categories capture.

4. **Variance Structure**: The relative weights reflect the empirical correlation structure in the data, revealing which proficiency categories actually distinguish between high and low-performing schools.

## Methodology Justification

The PCA-based weighting approach is justified for several reasons:

1. **Data-Driven**: Instead of imposing theoretical weights, the data itself reveals which categories are most important in practice.

2. **Maximizing Information**: The weights capture the categories that provide the most information about differences between schools.

3. **Reducing Noise**: By focusing on the strongest patterns, the influence of random variations or measurement errors is minimized.

4. **Empirical Validation**: The weights reflect actual patterns in student reading performance rather than assumptions about how reading skills should develop.

## Implications for Assessment

These findings suggest several important considerations for assessment frameworks:

1. **Equal vs. Differentiated Weighting**: The data strongly suggests that not all reading categories are equally important in explaining differences in school performance, challenging unweighted approaches.

2. **Context-Specific Development**: Reading skill development in the Philippine context may follow different patterns than assumed in standard models, possibly due to language factors, teaching approaches, or cultural contexts.

3. **Assessment Framework Alignment**: The current categories may need recalibration to better reflect how reading skills actually develop in the student population.

4. **Performance Measurement Accuracy**: Using data-derived weights may provide a more accurate reflection of reading proficiency than equal weighting across categories.

## Recommendations

Based on these findings, the following recommendations are proposed:

1. **Compare Analysis Results**: Evaluation of how school rankings and performance scores differ between standard and PCA-derived weights to understand the practical impact of these different weighting systems.

2. **Validation Studies**: Consideration of longitudinal studies to determine which weighting system better predicts future academic success or educational outcomes.

3. **Regional and Demographic Analysis**: Investigation into whether optimal weights vary by region, socioeconomic status, or other demographic factors.

4. **Assessment Framework Review**: Subject matter expert review of the current reading proficiency categories in light of these empirical findings.

5. **Mixed Methods Research**: Complementing this quantitative analysis with qualitative research to better understand the contextual factors influencing reading development patterns.

## Conclusion

The significant divergence between equal weighting and data-derived weights highlights the importance of empirical validation in educational assessment. While simple models provide valuable frameworks, they should be continually tested against actual learning data. This analysis represents a step toward more evidence-based approaches to measuring and supporting reading development in Philippine schools.

```python
# Code snippet showing how PCA weights were derived
pca_weights = crla_analyzer.optimize_weights_from_pca()
print("PCA-derived weights based on valid schools:")
for category, weight in pca_weights.items():
    print(f"  {category}: {weight}")
```

These findings invite a reconsideration of understanding reading proficiency assessment and how progress is measured in this critical educational domain.